# Notebook 03 | Sensitivity Analysis Across Market Inputs

Study how strategy rankings and execution-cost dispersion respond to changes in volatility, spread, ADV, and parent-order size. The objective is to identify where strategy preference is stable and where it becomes regime dependent.

## Scenario Grid

The sweep below varies core market and order parameters while holding the modeling structure fixed. Path counts are kept moderate for notebook interactivity; reporting-quality studies should use larger Monte Carlo samples.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

from execution_engine.analytics.sensitivity import run_sensitivity_analysis
from execution_engine.config import load_config
from execution_engine.visualization.plots_costs import plot_sensitivity_heatmap

config = load_config(PROJECT_ROOT / 'configs' / 'base.yaml')
parameter_grid = {
    'market.annualized_volatility': [0.18, 0.28, 0.42],
    'market.spread_bps': [4.0, 8.0, 14.0],
    'market.adv': [8_000_000, 12_000_000, 18_000_000],
    'order.parent_order_size': [150_000, 250_000, 400_000],
}
sensitivity = run_sensitivity_analysis(config, parameter_grid=parameter_grid, n_paths=20)
sensitivity.head()

## Sensitivity Heatmap

The heatmap focuses on mean implementation shortfall across volatility settings. Similar views can be produced for spread, ADV, and order size using the same output frame.

In [ ]:
focused = sensitivity[sensitivity['parameter'] == 'market.annualized_volatility']
fig, ax = plt.subplots(figsize=(10, 4))
plot_sensitivity_heatmap(focused, metric='mean_implementation_shortfall', ax=ax)
plt.tight_layout()
plt.show()

## Ranking Table

Review the ranked results directly to see where sensitivity is driven by the mean, by dispersion, or by changes in completion behavior.

In [ ]:
ranking_table = (
    sensitivity[['parameter', 'value', 'rank', 'strategy', 'mean_implementation_shortfall', 'std_implementation_shortfall']]
    .sort_values(['parameter', 'value', 'rank'])
)
ranking_table.head(20)